# ECSS Compliance Demo: ESSB-ST-U-007 Space Debris MitigationEnd-to-end verification that factpy can encode and audit ECSS compliance rules.Uses existing SDK/Service API — no core code changes.**Scenario:** A LEO mission "SENTINEL-7" with:- Disposal success probability: 92% (threshold: 90%) — confidence 0.85- Collision probability: 0.05% (threshold: 0.1%) — confidence 0.70- Passivation: all energy sources depleted — confidence 0.99**3 ECSS rules** encoded as Datalog:1. Disposal probability check (≥ threshold)2. Collision probability check (≤ threshold)3. Passivation status check (== "complete")**Full pipeline demonstrated:**Schema → Rules → Registry → Facts → Evaluate → Evidence Tree → Certainty→ Narrative → NL → **Souffle Provenance** → Audit → Static HTML

## 1. Schema: Mission entity with ECSS predicates

In [1]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path
from pprint import pprint

# Ensure src/ is on the Python path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import (
    SDKStore,
    Entity,
    Identity,
    Field,
    Rule,
    Pred,
    vars as sdk_vars,
)
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.domains.ecss import (
    extend_schema_ir_with_ecss_uncertainty_predicates,
    extend_schema_ir_with_ecss_vcd_predicates,
    ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
    ECSS_COMPLIANCE_STATUS_PRED_ID,
    ECSS_REQUIREMENT_PRED_ID,
    ECSS_VERIFICATION_METHOD_PRED_ID,
)
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session,
    close_runtime_session,
    reset_runtime_sessions_for_tests,
    write_runtime_fact,
    evaluate_runtime_derivation,
    explain_runtime_summary,
    explain_runtime_narrative,
    explain_runtime_nl,
    export_runtime_package,
    accept_runtime_derivation,
)
from factpy_kernel.audit import AuditQuery, load_audit_package

In [2]:
class Mission(Entity):
    mission_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    orbit_type: str = Field(cardinality="single")
    passivation_status: str = Field(cardinality="single")

sdk = SDKStore([Mission])

# Extend schema with ECSS uncertainty predicates (disposal/collision probability)
schema_ir = extend_schema_ir_with_ecss_uncertainty_predicates(sdk.schema_ir)
schema_ir = extend_schema_ir_with_ecss_vcd_predicates(schema_ir)
sdk = SDKStore([Mission], schema_ir=schema_ir)

print("=== Schema ready ===")
ecss_preds = [p["pred_id"] for p in sdk.schema_ir["predicates"] if p["pred_id"].startswith("ecss:")]
print(f"  ECSS predicates registered: {ecss_preds}")

=== Schema ready ===
  ECSS predicates registered: ['ecss:collision_probability_ppm', 'ecss:collision_probability_threshold_ppm', 'ecss:disposal_success_probability_ppm', 'ecss:disposal_success_threshold_ppm', 'ecss:requirement', 'ecss:verification_method', 'ecss:compliance_status', 'ecss:requirement_rid', 'ecss:review_milestone']


## 2. Rules: ESSB-ST-U-007 compliance checks

In [ ]:
# Rule 1: Disposal probability check
# "disposal success probability >= threshold"
with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    disposal_check_rule = Rule(
        id="q.essb_u007_disposal_check",
        version="1.0.0",
        select=[m, prob],
        where=[
            Pred(ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID, m, prob),
            Pred(ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID, m, threshold),
            prob >= threshold,
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.8,   # disposal probability importance
            "b0.a1": 0.5,   # threshold importance
        },
    )

# Rule 2: Collision probability check
# "collision probability <= threshold"
with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    collision_check_rule = Rule(
        id="q.essb_u007_collision_check",
        version="1.0.0",
        select=[m, prob],
        where=[
            Pred(ECSS_COLLISION_PROBABILITY_PPM_PRED_ID, m, prob),
            Pred(ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID, m, threshold),
            threshold >= prob,  # collision prob must be BELOW threshold
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.9,   # collision probability importance
            "b0.a1": 0.4,   # threshold importance
        },
    )

# Rule 3: Passivation check
# "mission passivation status is 'complete'"
with sdk_vars("m", "status") as (m, status):
    passivation_check_rule = Rule(
        id="q.essb_u007_passivation_check",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:passivation_status", m, status),
            status == "complete",
        ],
        expose=True,
        condition_weights={
            "b0.a0": 1.0,   # passivation is critical — full weight
        },
    )

print("=== Rules defined ===")
print(f"  {disposal_check_rule.id}: disposal success probability >= threshold")
print(f"  {collision_check_rule.id}: collision probability <= threshold")
print(f"  {passivation_check_rule.id}: passivation status == 'complete'")

## 3. Registry: register rules

In [ ]:
registry_dir = tempfile.mkdtemp(prefix="ecss_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
registry.register_rule_spec(sdk._compile_rule_input(disposal_check_rule))
registry.register_rule_spec(sdk._compile_rule_input(collision_check_rule))
registry.register_rule_spec(sdk._compile_rule_input(passivation_check_rule))

print(f"=== Registry ready: {registry_dir} ===")
print(f"  Rules registered: 3")

## 4. Data: SENTINEL-7 mission facts

In [5]:
with sdk.batch() as tx:
    s7 = tx.entity(Mission, mission_id="SENTINEL-7", locale="en")
    s7.name.set("Sentinel-7 LEO Observatory")
    s7.orbit_type.set("LEO")
    s7.passivation_status.set("complete")
    tx.commit()

mission_ref = sdk.ref(Mission, mission_id="SENTINEL-7", locale="en")

print(f"=== Mission entity created: {mission_ref} ===")

=== Mission entity created: idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq ===


## 5. Runtime: write facts + evaluate

In [6]:
reset_runtime_sessions_for_tests()
session_resp = open_runtime_session({"registry_root": registry_dir})
session_id = session_resp["session"]["session_id"]

# Disposal success probability: 92% = 920000 ppm
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    "e_ref": mission_ref,
    "rest_terms": [["int", 920000]],
    "meta": {"confidence": 0.85},
}, kind="add")

# Disposal threshold: 90% = 900000 ppm
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    "e_ref": mission_ref,
    "rest_terms": [["int", 900000]],
}, kind="add")

# Collision probability: 0.05% = 500 ppm
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    "e_ref": mission_ref,
    "rest_terms": [["int", 500]],
    "meta": {"confidence": 0.7},
}, kind="add")

# Collision threshold: 0.1% = 1000 ppm
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
    "e_ref": mission_ref,
    "rest_terms": [["int", 1000]],
}, kind="add")

print("=== Facts written ===")
print(f"  Disposal probability: 920000 ppm (92%), confidence=0.85")
print(f"  Disposal threshold:   900000 ppm (90%)")
print(f"  Collision probability: 500 ppm (0.05%), confidence=0.70")
print(f"  Collision threshold:   1000 ppm (0.1%)")

=== Facts written ===
  Disposal probability: 920000 ppm (92%), confidence=0.85
  Disposal threshold:   900000 ppm (90%)
  Collision probability: 500 ppm (0.05%), confidence=0.70
  Collision threshold:   1000 ppm (0.1%)


## 6. Evaluate: Disposal Success Probability Check

In [7]:
eval_disposal = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.disposal_check",
        "version": "1.0.0",
        "target": ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
        "head_vars": ["$m", "$prob"],
        "where": [
            ["ruleref", "q.essb_u007_disposal_check", "1.0.0", ["$m", "$prob"]],
        ],
        "mode": "native",
    }
})

if eval_disposal["ok"] and eval_disposal["evaluation"]["candidates"]:
    disposal_cand = eval_disposal["evaluation"]["candidates"][0]
    disposal_cid = disposal_cand["candidate_id"]
    print(f"  ✅ Disposal check PASSED")
    print(f"  candidate_id: {disposal_cid}")
    print(f"  confidence_kind: {disposal_cand['confidence_kind']}")

    # Summary
    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": disposal_cid})
    if summary["ok"]:
        print(f"\n  --- Evidence Tree Summary ---")
        s = summary["summary"]
        print(f"  support_kind: {s.get('support_kind')}")
        print(f"  witness_assertion_count: {s.get('witness_assertion_count')}")
        print(f"  rule_ref_count: {s.get('rule_ref_count')}")
        cs = summary.get("certainty_summary")
        if cs:
            print(f"\n  --- Certainty Summary ---")
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            print(f"  aggregation: {cs['aggregation']}")
            for c in cs["conditions"]:
                print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")

    # Narrative
    narrative = explain_runtime_narrative(session_id, {"kind": "candidate", "id": disposal_cid})
    if narrative["ok"]:
        narr = narrative["narrative"]
        print(f"\n  --- Narrative ---")
        for key in ["headline", "overview_lines", "evidence_lines", "rule_chain_lines", "certainty_lines"]:
            val = narr.get(key)
            if val:
                if isinstance(val, list):
                    for line in val:
                        print(f"  [{key}] {line}")
                else:
                    print(f"  [{key}] {val}")
else:
    print(f"  ❌ Disposal check FAILED or no candidates")
    print(f"  Response: {eval_disposal}")

  ✅ Disposal check PASSED
  candidate_id: cand_v2:a1b9a23288ff9c58ada4a6f36e7bc48af0968f3289470a950553b75dfd8ecf95
  confidence_kind: certainty

  --- Evidence Tree Summary ---
  support_kind: native_binding_v1
  witness_assertion_count: 2
  rule_ref_count: 1

  --- Certainty Summary ---
  aggregate_certainty: 0.5
  aggregation: bottleneck
    b0.a0: weight=0.8, impact=0.68
    b0.a1: weight=0.5, impact=0.5
    b0.a2: weight=None, impact=None

  --- Narrative ---
  [headline] Candidate cand_v2:a1b9a23288ff9c58ada4a6f36e7bc48af0968f3289470a950553b75dfd8ecf95 uses support kind native_binding_v1 across 12 tree node(s).
  [overview_lines] Root result kind: fact.
  [overview_lines] Role counts: structural=4, witness=4, constraint=2, rule_chain=2, terminal=0, degraded=0.
  [overview_lines] Recursive depth: 1.
  [evidence_lines] Witness assertions: 2.
  [evidence_lines] Witness nodes: 4; constraint nodes: 2.
  [rule_chain_lines] Rule reference nodes: 1.
  [rule_chain_lines] Recursive proof de

## 7. Evaluate: Collision Probability Check

In [8]:
eval_collision = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.collision_check",
        "version": "1.0.0",
        "target": ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
        "head_vars": ["$m", "$prob"],
        "where": [
            ["ruleref", "q.essb_u007_collision_check", "1.0.0", ["$m", "$prob"]],
        ],
        "mode": "native",
    }
})

if eval_collision["ok"] and eval_collision["evaluation"]["candidates"]:
    collision_cand = eval_collision["evaluation"]["candidates"][0]
    collision_cid = collision_cand["candidate_id"]
    print(f"  ✅ Collision check PASSED")
    print(f"  candidate_id: {collision_cid}")
    print(f"  confidence_kind: {collision_cand['confidence_kind']}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": collision_cid})
    if summary["ok"]:
        cs = summary.get("certainty_summary")
        if cs:
            print(f"\n  --- Certainty Summary ---")
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            for c in cs["conditions"]:
                print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")

    nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": collision_cid})
    if nl["ok"]:
        print(f"\n  --- NL Explanation ---")
        for i, p in enumerate(nl["explain_nl"]["paragraphs"], 1):
            print(f"  [{i}] {p[:150]}...")
else:
    print(f"  ❌ Collision check FAILED or no candidates")
    print(f"  Response: {eval_collision}")

  ✅ Collision check PASSED
  candidate_id: cand_v2:58edc72c01cc4f72f6ed4d1b470dea6d3733612d9601be39e6bbbab50f52724c
  confidence_kind: certainty

  --- Certainty Summary ---
  aggregate_certainty: 0.4
    b0.a0: weight=0.9, impact=0.63
    b0.a1: weight=0.4, impact=0.4
    b0.a2: weight=None, impact=None

  --- NL Explanation ---
  [1] Candidate cand_v2:58edc72c01cc4f72f6ed4d1b470dea6d3733612d9601be39e6bbbab50f52724c uses support kind native_binding_v1 across 12 tree node(s). Root re...
  [2] Evidence summary: Witness assertions: 2. Witness nodes: 4; constraint nodes: 2....
  [3] Rule-chain summary: Rule reference nodes: 1. Recursive proof depth: 1....
  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open referenced support branches to inspect recursive...
  [5] Certainty summary: Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.4. Condition b0.a1 (predicate_witness_group): weight=...


## 8. Evaluate: Passivation Check

In [ ]:
# Write passivation fact
write_runtime_fact(session_id, {
    "pred_id": "mission:passivation_status",
    "e_ref": mission_ref,
    "rest_terms": [["string", "complete"]],
    "meta": {"confidence": 0.99},
}, kind="add")

# Evaluate passivation check
eval_passivation = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.passivation_check",
        "version": "1.0.0",
        "target": "mission:passivation_status",
        "head_vars": ["$m", "$status"],
        "where": [
            ["ruleref", "q.essb_u007_passivation_check", "1.0.0", ["$m", "$status"]],
        ],
        "mode": "native",
    }
})

if eval_passivation["ok"] and eval_passivation["evaluation"]["candidates"]:
    pass_cand = eval_passivation["evaluation"]["candidates"][0]
    pass_cid = pass_cand["candidate_id"]
    print(f"  \u2705 Passivation check PASSED")
    print(f"  confidence_kind: {pass_cand['confidence_kind']}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": pass_cid})
    if summary["ok"]:
        cs = summary.get("certainty_summary")
        if cs:
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            for c in cs["conditions"]:
                if c["weight"] is not None:
                    print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")
else:
    print(f"  \u274c Passivation check FAILED")
    print(f"  Response: {eval_passivation}")

## 9. Souffle Provenance: Engine-Native Proof Tree

The evidence tree (§6-§8) shows rule structure + fact instances — but it's reconstructed
**after** Souffle finishes, from support artifacts. Souffle can also produce its own
**native proof tree** during execution via `-t explain`, which is significantly richer:

- Shows the full **assertion selection logic** (which assertion was chosen and why)
- Shows **negation leaves** (`NOT better_asrt` — no competing assertion exists)
- Shows **comparison evaluation** (`920000 >= 900000` as a leaf fact)
- Shows **subproof truncation markers** (depth-limited subtrees that can be drilled into)

This section demonstrates `run_package_provenance(...)` — an adapter-local caller
that re-runs Souffle with `-t explain` on a query-bearing exported package.

In [ ]:
from factpy_kernel.adapters.souffle.provenance import run_package_provenance
from factpy_kernel.adapters.souffle.runner import run_package

# Step 1: Export a query-bearing inference package for the disposal check.
disposal_compiled_where = sdk._compile_rule_input(disposal_check_rule)["where"]

provenance_pkg_dir = tempfile.mkdtemp(prefix="ecss_provenance_")
prov_export = export_runtime_package(session_id, {
    "out_dir": provenance_pkg_dir,
    "package_kind": "inference",
    "query": {
        "where": disposal_compiled_where,
        "query_rel": "disposal_check",
    },
})
assert prov_export["ok"], f"provenance export failed: {prov_export}"

# Step 2: Run the package to produce output tuples.
run_manifest_path = run_package(Path(provenance_pkg_dir), ["__query__"], engine="souffle")

# Step 3: Read the output to construct the provenance query.
out_path = Path(provenance_pkg_dir) / "outputs" / "disposal_check.out.facts"
output_content = out_path.read_text().strip()
row_parts = output_content.split("\n")[0].split("\t")
provenance_query = "disposal_check(" + ", ".join(f'\"{p}\"' for p in row_parts) + ")"
print(f"Provenance query: {provenance_query}")

# Step 4: Run Souffle provenance explain on the same package.
proof_trees = run_package_provenance(provenance_pkg_dir, [provenance_query])
print(f"Proof trees returned: {len(proof_trees)}")

# Step 5: Display the proof tree.
if proof_trees:
    tree = proof_trees[0]
    print(f"\n{'='*70}")
    print(f"SOUFFLE PROOF TREE: Disposal Success Probability Check")
    print(f"Query: {tree.query}")
    print(f"{'='*70}\n")

    def print_proof_node(node, indent=0):
        prefix = "  " * indent
        args_display = ", ".join(node.args[:3])
        if len(node.args) > 3:
            args_display += ", ..."
        label = f"{node.relation}({args_display})"

        if node.node_type == "axiom":
            print(f"{prefix}FACT: {label}")
        elif node.node_type == "negation":
            print(f"{prefix}NOT: {label}")
        elif node.node_type == "subproof":
            print(f"{prefix}SUBPROOF: {label}  (depth limit)")
        else:
            rule = node.rule_number or "?"
            print(f"{prefix}RULE {rule}: {label}")

        for child in node.children:
            print_proof_node(child, indent + 1)

    print_proof_node(tree.root)

    print(f"\n{'='*70}")
    print("KEY OBSERVATIONS:")
    print("  * claim_arg nodes show the ACTUAL assertion IDs that stored each value")
    print("  * chosen_asrt shows HOW the system selected the best assertion")
    print("  * NOT better_asrt proves no competing assertion exists")
    print("  * The >= comparison appears as a leaf fact")
    print("  * SUBPROOF nodes indicate depth-limited subtrees")
    print(f"{'='*70}")

## 10. Audit: export + round-trip

In [ ]:
# Accept all candidates (one at a time, singular "candidate" key)
all_evals = [
    ("disposal", eval_disposal),
    ("collision", eval_collision),
    ("passivation", eval_passivation),
]

for label, eval_resp in all_evals:
    if eval_resp["ok"] and eval_resp["evaluation"]["candidates"]:
        for cand in eval_resp["evaluation"]["candidates"]:
            resp = accept_runtime_derivation(session_id, {"candidate": cand})
            assert resp["ok"], f"accept {label} failed: {resp}"
            print(f"  \u2705 Accepted {label} candidate")

# Export audit package
audit_dir = tempfile.mkdtemp(prefix="ecss_audit_")
export_resp = export_runtime_package(session_id, {
    "out_dir": audit_dir,
    "package_kind": "audit",
})
assert export_resp["ok"], f"export failed: {export_resp}"
print(f"\n  Export: ok={export_resp['ok']}")

# Load and query
pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)

candidates = aq.list_candidates()
print(f"  Audit candidates: {len(candidates)}")

for cand_row in candidates:
    cid = cand_row["candidate_id"]

    # Evidence tree round-trip
    tree = aq.get_candidate_evidence_tree(cid)
    if tree:
        print(f"\n  --- Audit Evidence Tree for {cid[:30]}... ---")
        print(f"  root node_kind: {tree.get('root', {}).get('node_kind', '?')}")
        root_children = tree.get('root', {}).get('children', [])
        for child in root_children:
            nk = child.get('node_kind', '?')
            if nk == 'rule_ref_section':
                for rr in child.get('children', []):
                    print(f"    rule_ref: {rr.get('rule_ref_id', '?')} v{rr.get('rule_ref_version', '?')}")

    # Certainty round-trip
    cs = aq.get_candidate_certainty_summary(cid)
    if cs:
        print(f"  certainty: aggregate={cs['aggregate_certainty']}, aggregation={cs['aggregation']}")

    # Narrative round-trip
    narr = aq.get_candidate_evidence_tree_narrative(cid)
    if narr:
        cert_lines = narr.get("certainty_lines", [])
        if cert_lines:
            print(f"  narrative certainty_lines: {len(cert_lines)} lines")

print(f"\n  Audit package: {audit_dir}")

## 11. Static site

In [10]:
from factpy_kernel.audit.static_ui import render_audit_static_site

site_dir = tempfile.mkdtemp(prefix="ecss_site_")
render_audit_static_site(audit_dir, site_dir)

site_files = sorted(Path(site_dir).rglob("*.html"))
print(f"Static site: {len(site_files)} pages at {site_dir}")
for f in site_files[:10]:
    print(f"  {f.relative_to(site_dir)}")

Static site: 13 pages at /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/ecss_site__m88roxv
  assertions/132661eb8ffc4bbe976f883eae87ef67.html
  assertions/5a3a95c55036442da21e5b2d2a591873.html
  assertions/6a97045caf02428fb05c3131345dffc6.html
  assertions/c842f6188e59452896e3f9daba0c509d.html
  authoring_apply_events.html
  candidate_evidence.html
  compliance_matrix.html
  index.html
  indexes/error_classes.html
  indexes/event_kinds.html


## 12. Cleanup

In [11]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()

print("ECSS COMPLIANCE DEMO COMPLETE")
print(f"\nStatic audit site: {site_dir}")
print("Open candidate_evidence/*.html in a browser to see the evidence tree.")

ECSS COMPLIANCE DEMO COMPLETE

Static audit site: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/ecss_site__m88roxv
Open candidate_evidence/*.html in a browser to see the evidence tree.
